# ROGII - Advanced Modeling

This notebook is the first experiment after the baseline plateau. The current feature baseline reached the best public score of `15.491`, but the latest rerun did not improve it. The next useful direction is therefore new geological signal, not more tuning of the same rolling-feature model.

Experiment focus:

1. Keep the carry-forward baseline as the safety anchor.
2. Add typewell-aware `GR` alignment features around the carry-forward `TVT` estimate.
3. Train a residual tree model under held-out-well masked-tail validation.
4. Write `submission.csv` only from the advanced model when validation beats carry-forward.

The notebook is designed to run directly on Kaggle with the competition dataset attached.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 80)

RANDOM_STATE = 42
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
COMPETITION_SLUG = 'rogii-wellbore-geology-prediction'
DATA_ROOT_CANDIDATES = [
    KAGGLE_INPUT_ROOT / 'competitions' / COMPETITION_SLUG,
    KAGGLE_INPUT_ROOT / COMPETITION_SLUG,
]
WORK_DIR = Path('/kaggle/working')
SUBMISSION_PATH = WORK_DIR / 'submission.csv'


def resolve_data_root(candidates):
    for candidate in candidates:
        if (candidate / 'sample_submission.csv').exists():
            return candidate
    for sample_file in KAGGLE_INPUT_ROOT.rglob('sample_submission.csv') if KAGGLE_INPUT_ROOT.exists() else []:
        if COMPETITION_SLUG in sample_file.as_posix():
            return sample_file.parent
    return candidates[0]


DATA_ROOT = resolve_data_root(DATA_ROOT_CANDIDATES)
print('DATA_ROOT:', DATA_ROOT)
print('sample_submission exists:', (DATA_ROOT / 'sample_submission.csv').exists())


## 1. Load Files

This experiment uses both horizontal wells and typewells. The public test sample only has three wells, so the extra typewell features are cheap at inference time. During validation, the same feature code is applied to masked training wells so the experiment remains inference-safe.


In [ ]:
def find_files(root: Path, pattern: str):
    return sorted(root.rglob(pattern)) if root.exists() else []


def well_name_from_horizontal_path(path: Path) -> str:
    return path.name.split('__horizontal_well.csv')[0]


def well_name_from_typewell_path(path: Path) -> str:
    return path.name.split('__typewell.csv')[0]


def parse_submission_id(value):
    well, row = str(value).rsplit('_', 1)
    return well, int(row)


def get_column(df: pd.DataFrame, name: str):
    lookup = {col.lower(): col for col in df.columns}
    return lookup.get(name.lower())


train_files = find_files(DATA_ROOT / 'train', '*__horizontal_well.csv')
test_files = find_files(DATA_ROOT / 'test', '*__horizontal_well.csv')
train_typewell_files = find_files(DATA_ROOT / 'train', '*__typewell.csv')
test_typewell_files = find_files(DATA_ROOT / 'test', '*__typewell.csv')

train_typewell_lookup = {well_name_from_typewell_path(path): path for path in train_typewell_files}
test_typewell_lookup = {well_name_from_typewell_path(path): path for path in test_typewell_files}

sample_submission = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
id_col = sample_submission.columns[0]
target_col = 'tvt' if 'tvt' in sample_submission.columns else sample_submission.columns[-1]
parsed_ids = sample_submission[id_col].map(parse_submission_id)
sample_submission['well'] = [item[0] for item in parsed_ids]
sample_submission['row_idx'] = [item[1] for item in parsed_ids]

print('train horizontal wells:', len(train_files))
print('train typewells:', len(train_typewell_files))
print('test horizontal wells:', len(test_files))
print('test typewells:', len(test_typewell_files))
print('submission rows:', len(sample_submission))
display(sample_submission.head())


## 2. Baseline And Typewell-Alignment Features

The model still predicts residuals over carry-forward. The advanced part is a small TVT-offset search against the typewell `GR` curve:

- interpolate typewell `GR` at the carry-forward `TVT` estimate;
- test nearby TVT offsets around that estimate;
- record the best-matching offset and `GR` difference;
- add local typewell slope and context-spread features.

These features are available for train validation and for public test inference. They avoid train-only geology-top columns.


In [ ]:
ROLL_WINDOWS = (25, 101, 301)
TYPEWELL_OFFSETS = np.array([-200.0, -100.0, -50.0, 0.0, 50.0, 100.0, 200.0], dtype='float64')
FEATURE_COLUMNS = None
NON_FEATURE_COLUMNS = {'well', 'target_tvt', 'target_residual', 'tail_fraction'}
TYPEWELL_CACHE = {}


def numeric_col(df, name, default=np.nan):
    col = get_column(df, name)
    if col is None:
        return pd.Series(default, index=df.index, dtype='float64')
    return pd.to_numeric(df[col], errors='coerce').astype('float64')


def tvt_input_series(df):
    tvt_input_col = get_column(df, 'TVT_input')
    tvt_col = get_column(df, 'TVT')
    if tvt_input_col is not None:
        return pd.to_numeric(df[tvt_input_col], errors='coerce').astype('float64').reset_index(drop=True)
    if tvt_col is not None:
        return pd.to_numeric(df[tvt_col], errors='coerce').astype('float64').reset_index(drop=True)
    return pd.Series(np.nan, index=range(len(df)), dtype='float64')


def carry_forward_prediction(df):
    y_input = tvt_input_series(df)
    carry = y_input.ffill().bfill()
    if carry.isna().all():
        carry = pd.Series(0.0, index=range(len(df)), dtype='float64')
    return carry.astype('float64')


def add_rolling_features(out, source, prefix):
    for window in ROLL_WINDOWS:
        rolled = source.rolling(window=window, min_periods=1)
        out[f'{prefix}_roll_mean_{window}'] = rolled.mean()
        out[f'{prefix}_roll_std_{window}'] = rolled.std().fillna(0.0)
        out[f'{prefix}_roll_min_{window}'] = rolled.min()
        out[f'{prefix}_roll_max_{window}'] = rolled.max()
    return out


def clean_numeric_frame(out):
    numeric_features = [col for col in out.columns if col != 'well']
    out[numeric_features] = out[numeric_features].replace([np.inf, -np.inf], np.nan)
    medians = out[numeric_features].median(numeric_only=True)
    out[numeric_features] = out[numeric_features].fillna(medians).fillna(0.0)
    return out


def load_typewell_curve(path):
    if path is None:
        return None
    cache_key = str(path)
    if cache_key in TYPEWELL_CACHE:
        return TYPEWELL_CACHE[cache_key]

    df = pd.read_csv(path)
    tvt_col = get_column(df, 'TVT')
    gr_col = get_column(df, 'GR')
    if tvt_col is None or gr_col is None:
        TYPEWELL_CACHE[cache_key] = None
        return None

    curve = pd.DataFrame({
        'tvt': pd.to_numeric(df[tvt_col], errors='coerce'),
        'gr': pd.to_numeric(df[gr_col], errors='coerce'),
    }).dropna()
    curve = curve.sort_values('tvt').drop_duplicates('tvt')
    if len(curve) < 2:
        TYPEWELL_CACHE[cache_key] = None
        return None

    gr = curve['gr'].interpolate(limit_direction='both').to_numpy(dtype='float64')
    tvt = curve['tvt'].to_numpy(dtype='float64')
    TYPEWELL_CACHE[cache_key] = {'tvt': tvt, 'gr': gr}
    return TYPEWELL_CACHE[cache_key]


def interpolate_typewell_gr(curve, tvt_values):
    if curve is None:
        return np.full(len(tvt_values), np.nan, dtype='float64')
    return np.interp(tvt_values, curve['tvt'], curve['gr'], left=curve['gr'][0], right=curve['gr'][-1])


def add_typewell_alignment_features(out, gr_interp, carry, typewell_path):
    curve = load_typewell_curve(typewell_path)
    carry_values = carry.to_numpy(dtype='float64')
    gr_values = gr_interp.to_numpy(dtype='float64')

    if curve is None:
        out['typewell_available'] = 0
        out['typewell_gr_at_carry'] = 0.0
        out['typewell_gr_diff_at_carry'] = 0.0
        out['typewell_best_offset'] = 0.0
        out['typewell_best_abs_gr_diff'] = 0.0
        out['typewell_best_signed_gr_diff'] = 0.0
        out['typewell_context_gr_std'] = 0.0
        out['typewell_local_slope'] = 0.0
        return out

    candidate_gr = []
    for offset in TYPEWELL_OFFSETS:
        aligned_gr = interpolate_typewell_gr(curve, carry_values + offset)
        candidate_gr.append(aligned_gr)
        out[f'typewell_gr_offset_{int(offset):+d}'] = aligned_gr
        out[f'typewell_absdiff_offset_{int(offset):+d}'] = np.abs(gr_values - aligned_gr)

    candidate_gr = np.vstack(candidate_gr).T
    diffs = np.abs(candidate_gr - gr_values[:, None])
    best_idx = np.nanargmin(diffs, axis=1)
    rows = np.arange(len(out))
    best_gr = candidate_gr[rows, best_idx]

    out['typewell_available'] = 1
    out['typewell_gr_at_carry'] = candidate_gr[:, np.where(TYPEWELL_OFFSETS == 0.0)[0][0]]
    out['typewell_gr_diff_at_carry'] = gr_values - out['typewell_gr_at_carry']
    out['typewell_best_offset'] = TYPEWELL_OFFSETS[best_idx]
    out['typewell_best_abs_gr_diff'] = np.abs(gr_values - best_gr)
    out['typewell_best_signed_gr_diff'] = gr_values - best_gr
    out['typewell_context_gr_std'] = np.nanstd(candidate_gr, axis=1)

    gr_minus = interpolate_typewell_gr(curve, carry_values - 50.0)
    gr_plus = interpolate_typewell_gr(curve, carry_values + 50.0)
    out['typewell_local_slope'] = (gr_plus - gr_minus) / 100.0
    return out


def build_features(df, well, typewell_lookup):
    n = len(df)
    idx = pd.Series(np.arange(n), dtype='float64')
    denom = max(n - 1, 1)

    md = numeric_col(df, 'MD').reset_index(drop=True)
    x = numeric_col(df, 'X').reset_index(drop=True)
    y = numeric_col(df, 'Y').reset_index(drop=True)
    z = numeric_col(df, 'Z').reset_index(drop=True)
    gr_raw = numeric_col(df, 'GR').reset_index(drop=True)
    y_input = tvt_input_series(df)
    carry = carry_forward_prediction(df)

    gr_interp = gr_raw.interpolate(limit_direction='both').ffill().bfill()
    if gr_interp.isna().all():
        gr_interp = pd.Series(0.0, index=range(n), dtype='float64')

    known = y_input.notna()
    known_idx = pd.Series(np.where(known, idx, np.nan)).ffill().fillna(0.0)
    distance_from_known = (idx - known_idx).clip(lower=0)
    hidden_flag = (~known).astype('int8')

    tvt_diff = carry.diff().fillna(0.0)
    recent_slope = tvt_diff.rolling(window=101, min_periods=1).mean().fillna(0.0)
    recent_volatility = tvt_diff.rolling(window=101, min_periods=1).std().fillna(0.0)

    out = pd.DataFrame({
        'well': well,
        'row_idx': idx,
        'n_rows': float(n),
        'rel_pos': idx / denom,
        'distance_from_known': distance_from_known,
        'distance_from_known_frac': distance_from_known / denom,
        'hidden_flag': hidden_flag,
        'md': md,
        'md_rel': (md - md.min()) / (md.max() - md.min()) if md.notna().sum() > 1 and md.max() != md.min() else idx / denom,
        'x_centered': x - x.mean(),
        'y_centered': y - y.mean(),
        'z_centered': z - z.mean(),
        'gr': gr_raw,
        'gr_interp': gr_interp,
        'gr_missing': gr_raw.isna().astype('int8'),
        'gr_centered': gr_interp - gr_interp.mean(),
        'carry_tvt': carry,
        'tvt_recent_slope': recent_slope,
        'tvt_recent_volatility': recent_volatility,
    })

    out = add_rolling_features(out, gr_interp, 'gr')
    out = add_rolling_features(out, carry, 'carry_tvt')
    out = add_typewell_alignment_features(out, gr_interp, carry, typewell_lookup.get(well))
    return clean_numeric_frame(out)


def make_masked_frame(df, tail_fraction):
    tvt_col = get_column(df, 'TVT')
    if tvt_col is None:
        return None, None, None
    y_true = numeric_col(df, 'TVT').reset_index(drop=True)
    eval_start = int(len(df) * (1 - tail_fraction))
    masked = df.copy().reset_index(drop=True)
    masked['TVT_input'] = y_true.copy()
    masked.loc[eval_start:, 'TVT_input'] = np.nan
    return masked, y_true, eval_start


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))) if mask.any() else np.nan


def select_feature_columns(frame):
    return [col for col in frame.columns if col not in NON_FEATURE_COLUMNS]


def align_feature_frame(frame):
    aligned = frame.reindex(columns=FEATURE_COLUMNS, fill_value=0.0).copy()
    aligned = aligned.replace([np.inf, -np.inf], np.nan)
    return aligned.fillna(0.0)


if train_files:
    sample_path = train_files[0]
    sample_well = well_name_from_horizontal_path(sample_path)
    sample_df = pd.read_csv(sample_path)
    masked_df, y_true, eval_start = make_masked_frame(sample_df, tail_fraction=0.30)
    sample_features = build_features(masked_df, sample_well, train_typewell_lookup)
    print('sample well:', sample_well)
    print('sample feature shape:', sample_features.shape)
    display(sample_features.filter(regex='well|gr|typewell|carry|distance').head())


## 3. Build Advanced Validation Tables

Validation uses held-out wells and masked tail intervals. Each simulated hidden interval is sampled to keep runtime manageable on Kaggle CPU while still training on many wells and multiple tail lengths.


In [ ]:
TAIL_FRACTIONS = (0.20, 0.30, 0.40)
MAX_TRAIN_WELLS = 520
MAX_VALIDATION_WELLS = 160
MAX_ROWS_PER_WELL_FOLD = 900
VALIDATION_WELL_FRACTION = 0.20


def sample_eval_rows(features, y_true, eval_start, max_rows, random_state):
    eval_idx = np.arange(eval_start, len(features))
    if len(eval_idx) > max_rows:
        rng = np.random.default_rng(random_state)
        eval_idx = np.sort(rng.choice(eval_idx, size=max_rows, replace=False))
    sampled = features.iloc[eval_idx].copy()
    sampled['target_tvt'] = y_true.iloc[eval_idx].to_numpy()
    sampled['target_residual'] = sampled['target_tvt'] - sampled['carry_tvt']
    return sampled


def build_modeling_table(files, typewell_lookup, max_wells, tail_fractions, max_rows_per_fold, seed=RANDOM_STATE):
    frames = []
    for well_number, path in enumerate(files[:max_wells]):
        well = well_name_from_horizontal_path(path)
        df = pd.read_csv(path)
        for frac_number, tail_fraction in enumerate(tail_fractions):
            masked, y_true, eval_start = make_masked_frame(df, tail_fraction)
            if masked is None:
                continue
            features = build_features(masked, well, typewell_lookup)
            sampled = sample_eval_rows(
                features,
                y_true,
                eval_start,
                max_rows=max_rows_per_fold,
                random_state=seed + 1000 * well_number + frac_number,
            )
            sampled['tail_fraction'] = tail_fraction
            frames.append(sampled)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


rng = np.random.default_rng(RANDOM_STATE)
train_paths = np.array(train_files[:MAX_TRAIN_WELLS], dtype=object)
rng.shuffle(train_paths)
valid_size = max(1, int(len(train_paths) * VALIDATION_WELL_FRACTION))
valid_paths = list(train_paths[:valid_size])
model_train_paths = list(train_paths[valid_size:])

train_table = build_modeling_table(model_train_paths, train_typewell_lookup, MAX_TRAIN_WELLS, TAIL_FRACTIONS, MAX_ROWS_PER_WELL_FOLD)
valid_table = build_modeling_table(valid_paths, train_typewell_lookup, MAX_VALIDATION_WELLS, TAIL_FRACTIONS, MAX_ROWS_PER_WELL_FOLD)
FEATURE_COLUMNS = select_feature_columns(train_table)

print('model train wells:', len(model_train_paths))
print('validation wells:', len(valid_paths))
print('train table:', train_table.shape)
print('valid table:', valid_table.shape)
print('feature count:', len(FEATURE_COLUMNS))
display(train_table.head())


## 4. Train Typewell-Alignment Residual Model

The model predicts `target_tvt - carry_tvt`. This keeps the submission anchored to the safest known baseline while allowing typewell alignment to adjust the hidden interval when validation supports it.


In [ ]:
def fit_advanced_model(train_table):
    X = align_feature_frame(train_table)
    y = train_table['target_residual']
    try:
        model = HistGradientBoostingRegressor(
            loss='squared_error',
            learning_rate=0.045,
            max_iter=320,
            max_leaf_nodes=31,
            min_samples_leaf=35,
            l2_regularization=0.08,
            random_state=RANDOM_STATE,
        )
        model.fit(X, y)
        model_name = 'HistGradientBoostingRegressor'
    except Exception as exc:
        print('HistGradientBoostingRegressor failed; falling back to RandomForestRegressor:', repr(exc))
        model = RandomForestRegressor(
            n_estimators=180,
            max_depth=12,
            min_samples_leaf=18,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        model.fit(X, y)
        model_name = 'RandomForestRegressor'
    return model, model_name


model, model_name = fit_advanced_model(train_table)
print('model:', model_name)

valid_pred_residual = model.predict(align_feature_frame(valid_table))
valid_advanced_pred = valid_table['carry_tvt'].to_numpy() + valid_pred_residual
carry_rmse = rmse(valid_table['target_tvt'], valid_table['carry_tvt'])
advanced_rmse = rmse(valid_table['target_tvt'], valid_advanced_pred)

print('carry_forward validation RMSE:', carry_rmse)
print('typewell_alignment validation RMSE:', advanced_rmse)
print('delta RMSE:', advanced_rmse - carry_rmse)

valid_scored = valid_table[['well', 'tail_fraction', 'target_tvt', 'carry_tvt']].copy()
valid_scored['typewell_alignment'] = valid_advanced_pred
valid_summary = (
    valid_scored
    .groupby(['tail_fraction'])
    .apply(lambda x: pd.Series({
        'carry_rmse': rmse(x['target_tvt'], x['carry_tvt']),
        'typewell_alignment_rmse': rmse(x['target_tvt'], x['typewell_alignment']),
        'rows': len(x),
        'wells': x['well'].nunique(),
    }))
    .reset_index()
)
valid_summary['delta'] = valid_summary['typewell_alignment_rmse'] - valid_summary['carry_rmse']
display(valid_summary)

use_advanced_model = bool(advanced_rmse < carry_rmse)
print('use_advanced_model_for_submission:', use_advanced_model)


## 5. Inspect Feature Signal

A quick importance proxy helps decide whether typewell alignment is actually used. For tree models that expose feature importances this reports them directly. For histogram gradient boosting, the notebook instead reports correlation between features and residuals as a lightweight signal check.


In [ ]:
if hasattr(model, 'feature_importances_'):
    importance = pd.DataFrame({
        'feature': FEATURE_COLUMNS,
        'importance': model.feature_importances_,
    }).sort_values('importance', ascending=False)
else:
    corr_rows = []
    residual = train_table['target_residual'].to_numpy(dtype='float64')
    for feature in FEATURE_COLUMNS:
        values = train_table[feature].to_numpy(dtype='float64')
        if np.nanstd(values) == 0 or np.nanstd(residual) == 0:
            corr = 0.0
        else:
            corr = float(np.corrcoef(values, residual)[0, 1])
        corr_rows.append({'feature': feature, 'abs_correlation_with_residual': abs(corr), 'correlation_with_residual': corr})
    importance = pd.DataFrame(corr_rows).sort_values('abs_correlation_with_residual', ascending=False)

display(importance.head(25))
display(importance[importance['feature'].str.contains('typewell', case=False, regex=False)].head(20))


## 6. Refit And Generate Submission

If masked-tail validation improves over carry-forward, refit on more training wells and generate the advanced submission. If not, write a carry-forward fallback submission so the notebook remains safe to submit.


In [ ]:
if use_advanced_model:
    refit_table = build_modeling_table(
        train_files,
        train_typewell_lookup,
        max_wells=min(len(train_files), 720),
        tail_fractions=TAIL_FRACTIONS,
        max_rows_per_fold=MAX_ROWS_PER_WELL_FOLD,
    )
    FEATURE_COLUMNS = select_feature_columns(refit_table)
    model, model_name = fit_advanced_model(refit_table)
    print('refit model:', model_name)
    print('refit table:', refit_table.shape)
else:
    print('Validation did not beat carry-forward. Submission will use carry-forward fallback.')


def predict_test_well(df, well):
    features = build_features(df.reset_index(drop=True), well, test_typewell_lookup)
    carry = features['carry_tvt'].to_numpy()
    if use_advanced_model:
        residual = model.predict(align_feature_frame(features))
        pred = carry + residual
        known = tvt_input_series(df).notna().to_numpy()
        pred[known] = tvt_input_series(df).to_numpy()[known]
        return pd.Series(pred, index=range(len(df)), dtype='float64')
    return pd.Series(carry, index=range(len(df)), dtype='float64')


test_lookup = {well_name_from_horizontal_path(path): path for path in test_files}
well_predictions = {}
for well, path in test_lookup.items():
    df = pd.read_csv(path)
    well_predictions[well] = predict_test_well(df, well).reset_index(drop=True)

fallback = 0.0
non_empty = [pred.dropna().to_numpy() for pred in well_predictions.values() if pred.dropna().size]
if non_empty:
    fallback = float(np.nanmedian(np.concatenate(non_empty)))

submission = sample_submission[[id_col]].copy()
values = []
missing_wells = set()
for sub_id in sample_submission[id_col]:
    well, row_idx = parse_submission_id(sub_id)
    pred = well_predictions.get(well)
    if pred is None or len(pred) == 0:
        missing_wells.add(well)
        values.append(fallback)
    elif 0 <= row_idx < len(pred):
        values.append(float(pred.iloc[row_idx]))
    else:
        values.append(float(pred.iloc[-1]))

submission[target_col] = values
submission.to_csv(SUBMISSION_PATH, index=False)

print('wrote:', SUBMISSION_PATH)
print('selected submission model:', 'typewell_alignment' if use_advanced_model else 'carry_forward')
print('rows:', len(submission))
print('missing wells:', len(missing_wells))
display(submission.head())
display(submission[target_col].describe())


## 7. Readout

This is the first advanced experiment after the baseline plateau:

- if typewell alignment beats carry-forward in masked-tail validation, submit it;
- if it does not, keep the carry-forward fallback and treat the feature signal as insufficient;
- compare the public score against the current best `15.491`.

A useful result is not only a better score. If typewell features rank high but the public score does not move, the next experiment should improve the alignment search. If typewell features rank low, the next experiment should explore different geological context features rather than only increasing model complexity.
